In [ ]:
!pip install transformers datasets accelerate

# Importante
Gere um access token do hugging face
https://huggingface.co/docs/hub/en/security-tokens

E adicione no colab com o HF_TOKEN, na aba "Secrets" (Ícone de chave na barra lateral)



In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Dataset de emails - Gerado com ChatGPT
emails = [
    # ====================== IMPRODUTIVO ======================
    "Feliz Natal e um próspero Ano Novo a todos!",
    "Muito obrigado pela ajuda, funcionou perfeitamente!",
    "Parabéns pelo excelente trabalho realizado neste trimestre!",
    "Desejo uma ótima semana para todos.",
    "Bom dia! Espero que todos estejam bem.",
    "Obrigado pelo retorno rápido!",
    "Gostaria de parabenizar a equipe pelo esforço na entrega do projeto.",
    "Agradecemos pela parceria e confiança de sempre.",
    "Foi um prazer participar da reunião de hoje.",
    "Desejo boas festas e muito sucesso no próximo ano.",
    "Obrigado novamente pela atenção de todos.",
    "Que todos tenham um excelente final de semana.",
    "Abraços a toda a equipe!",
    "Fico feliz em ter colaborado neste projeto.",
    "Agradeço pelo convite para o evento.",
    "Espero que estejam todos bem.",
    "Desejo uma ótima sexta-feira.",
    "Obrigado pelo reconhecimento do trabalho.",
    "Parabéns pelo aniversário da empresa!",
    "Desejo sucesso nas próximas entregas.",
    "Muito grato pela oportunidade.",
    "Agradeço pela resposta rápida.",
    "Espero que possamos nos encontrar em breve.",
    "Desejo felicidade a todos os envolvidos.",
    "Bom descanso para o feriado prolongado.",
    "Foi muito bom participar da reunião.",
    "Parabéns aos colegas pelo esforço.",
    "Agradeço o retorno recebido.",
    "Desejo uma ótima tarde para todos.",
    "Obrigado pelo ótimo atendimento.",
    "Fico contente com o resultado.",
    "Parabéns pela conquista recente.",
    "Desejo tudo de bom para vocês.",
    "Obrigado pela lembrança.",
    "Espero que aproveitem o evento.",
    "Agradeço de coração pelo suporte.",
    "Desejo um feliz aniversário a todos os aniversariantes do mês.",
    "Parabéns pelos ótimos resultados do trimestre.",
    "Fico muito feliz com o reconhecimento.",
    "Obrigado pela oportunidade de falar na reunião.",
    "Desejo muita paz e saúde a todos.",
    "Parabéns à equipe de TI pelo esforço.",
    "Agradeço o tempo dedicado.",
    "Fico contente em ter ajudado.",
    "Obrigado pela mensagem positiva.",
    "Desejo um ótimo feriado.",
    "Parabéns pela excelente liderança.",
    "Fico agradecido pela lembrança.",
    "Desejo boa sorte a todos os envolvidos.",
    "Muito obrigado pelo suporte oferecido.",

    # ====================== PRODUTIVO ======================
    "Olá, gostaria de saber o status da minha solicitação de acesso ao sistema.",
    "Preciso que reenviem a fatura do mês de agosto, não consegui acessar no portal.",
    "Estou com dificuldades para acessar minha conta, aparece erro de autenticação.",
    "Favor atualizar o andamento do chamado #4567 aberto na semana passada.",
    "Preciso de suporte técnico para instalar o software de relatórios.",
    "Segue em anexo o contrato atualizado para análise.",
    "A integração com o sistema externo não está funcionando, poderiam verificar?",
    "Poderiam me confirmar se o pagamento referente à nota 123 já foi processado?",
    "Envio em anexo o arquivo solicitado na reunião de ontem.",
    "Preciso de ajuda com meu relatório de despesas.",
    "O boleto da fatura de setembro não foi recebido, poderiam reenviar?",
    "Estou enfrentando erro ao gerar relatórios no sistema.",
    "Favor liberar o acesso ao sistema de compras.",
    "Solicito atualização sobre a abertura da conta corporativa.",
    "O portal do cliente não está funcionando corretamente.",
    "Segue planilha em anexo para validação.",
    "Poderiam verificar o motivo do atraso na aprovação da nota fiscal?",
    "Necessito confirmação de recebimento do documento enviado ontem.",
    "Estou com dificuldades para redefinir minha senha.",
    "Solicito informações sobre o prazo da análise de crédito.",
    "Favor verificar porque o contrato não aparece no sistema.",
    "Envio documentação em anexo para prosseguir com o processo.",
    "O chamado #123 continua aberto sem resposta.",
    "Preciso do relatório atualizado de vendas do último trimestre.",
    "Poderiam checar o erro de autenticação no servidor?",
    "Favor confirmar se a transferência bancária foi realizada.",
    "Preciso atualizar meu cadastro no sistema, como proceder?",
    "Segue o arquivo de comprovante em anexo.",
    "Estou enfrentando instabilidade no portal de fornecedores.",
    "Necessito o extrato bancário referente a setembro.",
    "Favor informar sobre a homologação da nova versão do sistema.",
    "Estou com problemas para acessar o e-mail corporativo.",
    "Solicito suporte para reinstalar o aplicativo de relatórios.",
    "Segue documento digitalizado para assinatura.",
    "Preciso saber se o processo de pagamento já foi concluído.",
    "Estou enfrentando lentidão ao acessar os relatórios.",
    "Favor confirmar recebimento do contrato assinado.",
    "O sistema apresenta erro ao carregar o dashboard.",
    "Preciso de ajuda para cadastrar novo usuário no sistema.",
    "Segue solicitação formal em anexo para validação.",
    "Poderiam informar a data de liberação da verba?",
    "Estou com dificuldades para abrir chamados no portal.",
    "Solicito acesso temporário ao sistema financeiro.",
    "O arquivo enviado não foi recebido, poderiam confirmar?",
    "Preciso que corrijam o erro de cálculo na fatura.",
    "Estou aguardando retorno sobre o pedido #999.",
    "Segue declaração necessária para prosseguimento.",
    "Necessito suporte imediato para desbloqueio de conta.",
    "Poderiam atualizar a situação do processo de auditoria?",
    "Segue solicitação em anexo para análise."
]

labels = (
    [0]*50 +  # 50 improdutivos
    [1]*50    # 50 produtivos
)

dataset = Dataset.from_dict({'text': emails, 'label': labels})
dataset = dataset.shuffle(seed=42)  # Embaralhar os dados do dataset. Seed para reprodutibilidade

# Separando os dados de treino e teste
dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
# Nome do modelo e tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Função de pré-processamento
def preprocess(examples):
  return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)

# Aplicar pré-processamento no dataset
dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
# Instancia o modelo. Será binário, pois classifica entre Produtivo ou Improdutivo
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Treinamento do modelo

# Métricas do treino
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# Configurações do treinamento
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# Salva o modelo
model.save_pretrained("./email_classifier_hf")
tokenizer.save_pretrained("./email_classifier_hf")

/tmp/ipython-input-2113818246.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


('./email_classifier_hf/tokenizer_config.json',
 './email_classifier_hf/special_tokens_map.json',
 './email_classifier_hf/vocab.txt',
 './email_classifier_hf/added_tokens.json',
 './email_classifier_hf/tokenizer.json')

In [ ]:
!zip -r email_classifier_hf.zip ./email_classifier_hf
from google.colab import files
files.download("email_classifier_hf.zip")

  adding: email_classifier_hf/ (stored 0%)
  adding: email_classifier_hf/model.safetensors (deflated 8%)
  adding: email_classifier_hf/tokenizer_config.json (deflated 75%)
  adding: email_classifier_hf/config.json (deflated 45%)
  adding: email_classifier_hf/special_tokens_map.json (deflated 42%)
  adding: email_classifier_hf/tokenizer.json (deflated 71%)
  adding: email_classifier_hf/vocab.txt (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>